In [1]:
import os
import yaml
import pickle
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split, cross_val_predict
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import gc
import json

# ============================================================
# [실험 1] 잔차 학습 (Residual Learning)
#   최종 예측 = 공식 계산값 + 모델이 예측한 잔차
#
# [실험 2] 동적 가중 블렌딩 (Dynamic Weighted Blending)
#   duration에 따라 RF/LR 가중치 자동 조절
#   300초 이하 -> RF 100%
#   3600초(1시간) -> LR 100%
# ============================================================

In [2]:
# ============================================================
# Config 로드
# ============================================================
CONFIG_PATH = "/content/config.yaml"
with open(CONFIG_PATH, 'r') as f:
    config = yaml.safe_load(f)

DRIVE_ROOT     = f"/content/drive/MyDrive/{config['project_name']}"
PROCESSED_PATH = os.path.join(DRIVE_ROOT, config['paths']['processed_data'],
                               config['paths']['processed_file'])
MODEL_PATH     = os.path.join(DRIVE_ROOT, config['paths']['models'])
REPORT_PATH    = os.path.join(DRIVE_ROOT, config['paths']['outputs'], "reports")

In [3]:
# ============================================================
# 데이터 로드 + 샘플링 + 증강
# ============================================================
df_full = pd.read_parquet(PROCESSED_PATH)

SAMPLE_SIZE = 2_000_000
df = df_full.sample(n=SAMPLE_SIZE, random_state=config['model']['random_state'])
del df_full; gc.collect()

DURATION_MULTIPLIERS = [1, 2, 4, 6, 12]
augmented_dfs = []
for m in DURATION_MULTIPLIERS:
    df_aug = df.copy()
    df_aug['duration']   = df_aug['duration']   * m
    df_aug['energy_kwh'] = df_aug['energy_kwh'] * m
    augmented_dfs.append(df_aug)

df_aug = pd.concat(augmented_dfs, ignore_index=True)
del augmented_dfs; gc.collect()
print(f"Augmented rows: {len(df_aug):,}")

Augmented rows: 10,000,000


In [5]:
# ============================================================
# 잔차 계산 (실험 1용)
# formula_pred = power_w * duration / 3600 / 1000
# residual = actual_energy - formula_pred
# ============================================================
# df_aug['formula_power'] = 200 + df_aug['cpu'] * 300 + df_aug['memory'] * 50
# df_aug['residual_power'] = df_aug['power_w'] - df_aug['formula_power']
# df_aug['residual_per_sec'] = df_aug['residual_power'] / df_aug['duration']

# print(f"\n[power_w 잔차 통계]")
# print(f"  mean : {df_aug['residual_power'].mean():.6f}")
# print(f"  std  : {df_aug['residual_power'].std():.6f}")



In [4]:
# ============================================================
# Train/Test 분할
# ============================================================
features = ["cpu", "memory", "duration"]
target   = "energy_kwh"

X = df_aug[features]
y = df_aug[target]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=config['model']['test_size'],
    random_state=config['model']['random_state']
)

del df_aug; gc.collect()
print(f"Train: {len(X_train):,} | Test: {len(X_test):,}")


Train: 8,000,000 | Test: 2,000,000


In [5]:
# ============================================================
# 베이스 모델 학습 
# ============================================================
rf_base = RandomForestRegressor(
    n_estimators=100,
    max_depth=15,
    min_samples_split=10,
    min_samples_leaf=5,
    random_state=config['model']['random_state'],
    n_jobs=-1
)
lr_base = LinearRegression()

rf_base.fit(X_train, y_train)
lr_base.fit(X_train, y_train)
print("Base models trained!")


Base models trained!


In [7]:
# ============================================================
# [실험 1] 잔차 학습 (Residual Learning)
# ============================================================
print("\n" + "="*50)
print("[실험 1] 잔차 학습")
print("="*50)

rf_res = RandomForestRegressor(
    n_estimators=100,
    max_depth=15,
    min_samples_split=10,
    min_samples_leaf=5,
    random_state=config['model']['random_state'],
    n_jobs=-1
)
rf_res.fit(X_train, y_train)
print("Residual RF trained!")

#  RF 예측 오차 계산
rf_pred_train  = rf_res.predict(X_train)
residual_train = y_train.values - rf_pred_train

# LR이 RF 오차 학습
lr_res = LinearRegression()
lr_res.fit(X_train, residual_train)

# 최종 예측 = RF 예측 + LR 오차 예측
rf_pred_test = rf_res.predict(X_test)
lr_pred_test = lr_res.predict(X_test)
final_pred_1 = rf_pred_test + lr_pred_test

rmse_1 = np.sqrt(mean_squared_error(y_test, final_pred_1))
mae_1  = mean_absolute_error(y_test, final_pred_1)
r2_1   = r2_score(y_test, final_pred_1)
mape_1 = np.mean(np.abs((y_test.values - final_pred_1) / y_test.values)) * 100

print(f"  RMSE : {rmse_1:.8f}")
print(f"  MAE  : {mae_1:.8f}")
print(f"  R2   : {r2_1:.8f}")
print(f"  MAPE : {mape_1:.4f}%")


[실험 1] 잔차 학습
Residual RF trained!
  RMSE : 0.00004344
  MAE  : 0.00001510
  R2   : 0.99999957
  MAPE : 0.3433%


In [8]:
# 4. 최종 평가
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

rf_pred_test = rf_res.predict(X_test)
lr_pred_test = rf_res.predict(X_test)
final_pred   = rf_pred_test + lr_pred_test

rmse = np.sqrt(mean_squared_error(y_test, final_pred))
r2   = r2_score(y_test, final_pred)
mape = np.mean(np.abs((y_test.values - final_pred) / y_test.values)) * 100

print(f"\n[Residual Learning 성능]")
print(f"  RMSE : {rmse:.8f}")
print(f"  R2   : {r2:.8f}")
print(f"  MAPE : {mape:.4f}%")

# 5. 저장
rf_file = os.path.join(MODEL_PATH, config['model']['model_names']['randomforest'])
lr_file = os.path.join(MODEL_PATH, config['model']['model_names']['linearregression'])

with open(rf_file, 'wb') as f: pickle.dump(rf_res, f)
with open(lr_file, 'wb') as f: pickle.dump(rf_res, f)

print(f"\nSaved RF : {rf_file} ({os.path.getsize(rf_file)/(1024**2):.1f} MB)")
print(f"Saved LR : {lr_file} ({os.path.getsize(lr_file)/(1024**2):.1f} MB)")
print("Done!")


[Residual Learning 성능]
  RMSE : 0.09542245
  R2   : -1.05642261
  MAPE : 100.0000%

Saved RF : /content/drive/MyDrive/EcoTracing/models/energy_model_randomforest.pkl (233.2 MB)
Saved LR : /content/drive/MyDrive/EcoTracing/models/energy_model_linearregression.pkl (233.2 MB)
Done!


In [9]:
import json

result_file = os.path.join(DRIVE_ROOT, config['paths']['outputs'], "reports",
                           config['model']['results']['results_json'])

with open(result_file, 'r') as f:
    results_json = json.load(f)

results_json['residual_learning'] = {
    'method'      : 'Residual Learning (RF + LR residual)',
    'description' : 'final = RF.predict(X) + LR.predict(RF_residual)',
    'rf_params'   : {
        'n_estimators'     : 100,
        'max_depth'        : 15,
        'min_samples_split': 10,
        'min_samples_leaf' : 5,
    },
    'features'    : features,
    'rmse'        : float(rmse),
    'r2'          : float(r2),
    'mape'        : float(mape),
}
results_json['best_model'] = 'residual_learning'

with open(result_file, 'w') as f:
    json.dump(results_json, f, indent=2, ensure_ascii=False)

print(f"Results updated: {result_file}")
print("Done!")

Results updated: /content/drive/MyDrive/EcoTracing/outputs/reports/phase1_full_results.json
Done!


In [6]:
# ============================================================
# [실험 2] 동적 가중 블렌딩 (Dynamic Weighted Blending)
# ============================================================
print("\n" + "="*50)
print("[실험 2] 동적 가중 블렌딩")
print("="*50)

TRAIN_DURATION_MAX = 300.0
MAX_DURATION = 3600.0
TRAIN_DURATION_SEC = 300

def dynamic_weight_rf(duration_sec):
    """
    RF 가중치 동적 계산
    - duration <= 300초 : RF = 1.0
    - 300 < duration <= 3600초: 선형 감소
    - duration > 3600초 : LR = 1.0
    """
    if duration_sec <= TRAIN_DURATION_MAX:
        return 1.0
    return max(0.0, 1.0 - (duration_sec - TRAIN_DURATION_MAX) / (MAX_DURATION - TRAIN_DURATION_MAX))

print("\n[duration별 RF/LR 가중치]")
for dur in [300, 600, 1200, 1800, 3600]:
    rf_w = dynamic_weight_rf(dur)
    print(f"  {dur:5d}초 ({dur/3600:.2f}h) -> RF={rf_w:.2f}, LR={1-rf_w:.2f}")

# 배치 가중치 계산
durations  = X_test['duration'].values
rf_weights = np.array([dynamic_weight_rf(d) for d in durations])
lr_weights = 1.0 - rf_weights

# RF: 300초 고정 + 스케일링
X_test_300 = X_test.copy()
X_test_300['duration'] = TRAIN_DURATION_SEC
rf_pred = rf_base.predict(X_test_300) * (durations / TRAIN_DURATION_SEC)

# LR: duration 직접 전달
lr_pred = lr_base.predict(X_test)

final_pred_2 = rf_weights * rf_pred + lr_weights * lr_pred

rmse_2 = np.sqrt(mean_squared_error(y_test, final_pred_2))
mae_2  = mean_absolute_error(y_test, final_pred_2)
r2_2   = r2_score(y_test, final_pred_2)
mape_2 = np.mean(np.abs((y_test.values - final_pred_2) / y_test.values)) * 100

print(f"\n  RMSE : {rmse_2:.8f}")
print(f"  MAE  : {mae_2:.8f}")
print(f"  R2   : {r2_2:.8f}")
print(f"  MAPE : {mape_2:.4f}%")



[실험 2] 동적 가중 블렌딩

[duration별 RF/LR 가중치]
    300초 (0.08h) -> RF=1.00, LR=0.00
    600초 (0.17h) -> RF=0.91, LR=0.09
   1200초 (0.33h) -> RF=0.73, LR=0.27
   1800초 (0.50h) -> RF=0.55, LR=0.45
   3600초 (1.00h) -> RF=0.00, LR=1.00

  RMSE : 0.00072097
  MAE  : 0.00019585
  R2   : 0.99988261
  MAPE : 0.1427%


In [ ]:
# ============================================================
# 최종 비교
# ============================================================
print("\n" + "="*50)
print("=== 최종 비교 ===")
print("="*50)
compare = pd.DataFrame([
    {"method": "Residual Learning",        "rmse": rmse_1, "r2": r2_1, "mape": mape_1},
    {"method": "Dynamic Weighted Blending","rmse": rmse_2, "r2": r2_2, "mape": mape_2},
])
print(compare.to_string(index=False))
# ============================================================
# 최적 모델 저장 (MAPE 기준)
# ============================================================
best = 1 if mape_1 <= mape_2 else 2
print(f"\n최적 방법: 실험 {best} - {'Residual Learning' if best == 1 else 'Dynamic Weighted Blending'}")

rf_file = os.path.join(MODEL_PATH, config['model']['model_names']['randomforest'])
lr_file = os.path.join(MODEL_PATH, config['model']['model_names']['linearregression'])

if best == 1:
    with open(rf_file, 'wb') as f: pickle.dump(rf_res, f)
    print(f"Saved RF (residual) : {rf_file} ({os.path.getsize(rf_file)/(1024**2):.1f} MB)")
else:
    with open(rf_file, 'wb') as f: pickle.dump(rf_base, f)
    with open(lr_file, 'wb') as f: pickle.dump(lr_base, f)
    print(f"Saved RF : {rf_file} ({os.path.getsize(rf_file)/(1024**2):.1f} MB)")
    print(f"Saved LR : {lr_file} ({os.path.getsize(lr_file)/(1024**2):.1f} MB)")


# ============================================================
# phase1_full_results.json 업데이트
# ============================================================
result_file = os.path.join(REPORT_PATH, config['model']['results']['results_json'])
with open(result_file, 'r') as f:
    results_json = json.load(f)

results_json['advanced_ensemble'] = {
    'residual_learning': {
        'method'      : 'Residual Learning',
        'description': 'final = RF.predict(X) + LR.predict(RF_residual)',
        'features'    : features,
        'rmse'        : float(rmse_1),
        'mae'         : float(mae_1),
        'r2'          : float(r2_1),
        'mape'        : float(mape_1),
    },
    'dynamic_blending': {
        'method'             : 'Dynamic Weighted Blending (RF+LR)',
        'description'        : 'duration <= 300s: RF=1.0 / duration >= 3600s: LR=1.0 (linear)',
        'train_duration_max' : TRAIN_DURATION_MAX,
        'max_duration'       : MAX_DURATION,
        'features'           : features,
        'rmse'               : float(rmse_2),
        'mae'                : float(mae_2),
        'r2'                 : float(r2_2),
        'mape'               : float(mape_2),
    },
    'best_method': f'실험 {best} - {"Residual Learning" if best == 1 else "Dynamic Weighted Blending"}'
}

with open(result_file, 'w') as f:
    json.dump(results_json, f, indent=2, ensure_ascii=False)

print(f"\nResults updated: {result_file}")
print("Done!")


=== 최종 비교 ===
                   method     rmse       r2     mape
        Residual Learning 0.000043 1.000000 0.343309
Dynamic Weighted Blending 0.000721 0.999883 0.142689

최적 방법: 실험 2 - Dynamic Weighted Blending
Saved RF : /content/drive/MyDrive/EcoTracing/models/energy_model_randomforest.pkl (233.2 MB)
Saved LR : /content/drive/MyDrive/EcoTracing/models/energy_model_linearregression.pkl (0.0 MB)

Results updated: /content/drive/MyDrive/EcoTracing/outputs/reports/phase1_full_results.json
Done!
